In [ ]:
# ===============================
# Phase 1: Data Architecture
# ===============================

import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Generate dataset
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=12,
    n_redundant=5,
    n_classes=2,
    weights=[0.9, 0.1],
    random_state=42
)


columns = [f'Feature_{i}' for i in range(20)]
df = pd.DataFrame(X, columns=columns)
df['Target'] = y

print(df['Target'].value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,   # maintain imbalance ratio
    random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Target
0    897
1    103
Name: count, dtype: int64


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# Baseline model
rf_baseline = RandomForestClassifier(random_state=42)
rf_baseline.fit(X_train_scaled, y_train)

# Predictions
y_pred = rf_baseline.predict(X_test_scaled)

# Metrics
baseline_acc = accuracy_score(y_test, y_pred)
baseline_f1 = f1_score(y_test, y_pred)

print("Baseline Accuracy:", baseline_acc)
print("Baseline F1 Score:", baseline_f1)

Baseline Accuracy: 0.91
Baseline F1 Score: 0.25


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

# Grid Search - Accuracy
grid_acc = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)

grid_acc.fit(X_train_scaled, y_train)

# Grid Search - F1
grid_f1 = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1
)

grid_f1.fit(X_train_scaled, y_train)

print("Best Params (Accuracy):", grid_acc.best_params_)
print("Best Score (Accuracy):", grid_acc.best_score_)

print("\nBest Params (F1):", grid_f1.best_params_)
print("Best Score (F1):", grid_f1.best_score_)

Best Params (Accuracy): {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 200}
Best Score (Accuracy): 0.92875

Best Params (F1): {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 200}
Best Score (F1): 0.45396825396825397


In [ ]:
import time

start = time.time()

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

grid_time = time.time() - start

print("Grid Search Time:", grid_time)
print("Best F1 (Grid):", grid_search.best_score_)

Grid Search Time: 69.70661759376526
Best F1 (Grid): 0.45396825396825397


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': np.arange(10, 500),
    'max_depth': [None] + list(np.arange(5, 30)),
    'min_samples_split': np.arange(2, 15)
}

start = time.time()

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train_scaled, y_train)

random_time = time.time() - start

print("Randomized Search Time:", random_time)
print("Best F1 (Random):", random_search.best_score_)

Randomized Search Time: 78.97929000854492
Best F1 (Random): 0.42852123721688945


In [ ]:
results = pd.DataFrame({
    "Method": ["Grid Search", "Randomized Search"],
    "Time Taken (sec)": [grid_time, random_time],
    "Best F1 Score": [
        grid_search.best_score_,
        random_search.best_score_
    ]
})

print(results)

              Method  Time Taken (sec)  Best F1 Score
0        Grid Search         69.706618       0.453968
1  Randomized Search         78.979290       0.428521
